In [2]:
!pip install recmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.2/13.2 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 292.8/292.8 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.9/235.9 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.8/385.8 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.5/133.5 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.7/59.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 744.4/744.4 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import files

In [ ]:
# Step 1: Upload datasets
uploaded = files.upload()

In [ ]:
# Load the datasets (the uploaded files will be available as their filenames)
sessions = pd.read_csv('train_sessions.csv')  # Replace with actual filename if different
purchases = pd.read_csv('train_purchases.csv')
item_features = pd.read_csv('item_features.csv')
candidate_items = pd.read_csv('candidate_items.csv')  # File containing candidate items

In [ ]:
# Step 2: Data Preprocessing
# Merge sessions with purchases and truncate at the first purchase
purchases = purchases.rename(columns={'item_id': 'purchased_item'})
data = sessions.merge(purchases, on='session_id', how='left')
data = data[data['timestamp'] < data.groupby('session_id')['timestamp'].transform('max')]

In [ ]:
# Merge with item features
data = data.merge(item_features, on='item_id', how='left')

In [ ]:
# Handle missing values
data.fillna({'feature_category_id': -1, 'feature_value_id': -1}, inplace=True)

In [ ]:
# One-hot encoding categorical features
encoder = OneHotEncoder(sparse=False, handle_unknown='ignore')
categorical_cols = ['feature_category_id', 'feature_value_id']
encoded_features = encoder.fit_transform(data[categorical_cols])
encoded_feature_names = encoder.get_feature_names_out(categorical_cols)
encoded_features_df = pd.DataFrame(encoded_features, columns=encoded_feature_names)
data = pd.concat([data.reset_index(drop=True), encoded_features_df], axis=1)
data.drop(columns=categorical_cols, inplace=True)

In [ ]:
# Normalize numerical features
scaler = StandardScaler()
num_cols = ['session_length', 'views']  # Example numerical columns
data[num_cols] = scaler.fit_transform(data[num_cols])

In [ ]:
# Ensure candidate items are used
candidate_items = candidate_items['item_id'].unique()
data = data[data['item_id'].isin(candidate_items)]

In [ ]:
# Step 3: Build Item-Item Similarity Matrix
item_feature_matrix = data.groupby('item_id').mean().drop(['session_id', 'timestamp', 'purchased_item'], axis=1)
similarity_matrix = cosine_similarity(item_feature_matrix)
similarity_df = pd.DataFrame(similarity_matrix, index=item_feature_matrix.index, columns=item_feature_matrix.index)

In [ ]:
# Step 4: Generate Recommendations
def get_top_n_recommendations(session_data, similarity_df, n=100):
    recommendations = {}
    for session_id, group in session_data.groupby('session_id'):
        viewed_items = group['item_id'].unique()
        scores = similarity_df.loc[viewed_items].mean(axis=0).sort_values(ascending=False)
        scores = scores[~scores.index.isin(viewed_items)]  # Exclude already viewed items
        recommendations[session_id] = scores.head(n).index.tolist()
    return recommendations

In [ ]:
# Generate top-n recommendations for each session
top_n = get_top_n_recommendations(data, similarity_df, n=100)

In [ ]:
# Step 5: Evaluate Recommendations with Recmetrics
all_items = set(candidate_items)
predicted_items = {session_id: items for session_id, items in top_n.items()}
actual_items = {session_id: [item] for session_id, item in purchases[['session_id', 'purchased_item']].values}

In [ ]:
# Mean Reciprocal Rank (MRR)
def mean_reciprocal_rank(actual_items, predicted_items):
    mrr = 0
    count = 0
    for session_id, true_items in actual_items.items():
        pred_items = predicted_items.get(session_id, [])
        for rank, item in enumerate(pred_items, 1):
            if item in true_items:
                mrr += 1 / rank
                count += 1
                break
    return mrr / count if count > 0 else 0

In [ ]:
# Coverage: proportion of candidate items recommended at least once
def coverage(predicted_items, all_items):
    recommended_items = set([item for items in predicted_items.values() for item in items])
    return len(recommended_items) / len(all_items)

In [ ]:
# Personalization
personalization_score = personalization(predicted_items)
print(f"Personalization: {personalization_score}")

In [ ]:
# Novelty
novelty_score = novelty(predicted_items, all_items)
print(f"Novelty: {novelty_score}")

In [ ]:
# Step 6: Save Recommendations
submission = []
for session_id, items in top_n.items():
    for rank, item_id in enumerate(items):
        submission.append([session_id, item_id, rank + 1])

submission_df = pd.DataFrame(submission, columns=['session_id', 'item_id', 'rank'])
submission_df.sort_values(by=['session_id', 'rank'], inplace=True)

In [ ]:
# Save to a CSV file
submission_df.to_csv('submission.csv', index=False)

In [ ]:
# Step 7: Download the results
files.download('submission.csv')